# Part III — The Transformer, Training, and Inference

**The climax of the foundations.**

In Part I you learned how text becomes numbers. In Part II you learned how attention lets tokens talk to each other. Now we assemble the machine.

By the end of this notebook, you will have:

1. Built a complete transformer block from scratch (~50 lines of PyTorch)
2. Understood why residual connections are the single most important trick
3. Compared three families of positional encoding (sinusoidal, RoPE, ALiBi)
4. Traced a token through every layer of a real LLM architecture
5. **Trained a nanoGPT on Shakespeare** — yes, actually trained it
6. **Generated text** with greedy, temperature, top-k, and top-p sampling

This is a long one. Budget 60–75 minutes. Bring coffee.

---

### The map

```
data -> tokenizer -> embeddings -> [transformer block x N] -> logits
                                        ^                      |
                                        |                      v
                                   (Part III)             next token
                                                               |
                                                               v
                                                         (loop for generation)
```

Everything before "transformer block" is Parts I–II. Everything inside the block, plus the training loop and the generation loop, is Part III.

Let's go.

In [ ]:
# Install deps if needed. Uncomment on a fresh environment.
# !pip install torch matplotlib numpy tiktoken

import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

torch.manual_seed(1337)
np.random.seed(1337)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

---
# Section 1 — The Full Transformer Block

## The recipe that ate the world

A transformer block, stripped to its essence, is just four ingredients:

1. **Multi-head attention** (tokens mix with each other)
2. **Feed-forward network** (each token thinks to itself)
3. **Residual connections** (the "just let information flow through" trick)
4. **Layer normalization** (keep activations from exploding)

That's it. Stack this block 12, 32, or 96 times and you have GPT-2, Llama, or GPT-4.

### The modern layout: Pre-Norm

The original 2017 transformer used **post-norm**: normalize *after* adding the residual. This turned out to be training-unstable at depth. Everyone now uses **pre-norm**:

$$
\begin{aligned}
x &\leftarrow x + \text{MultiHead}(\text{LN}(x)) \\
x &\leftarrow x + \text{FFN}(\text{LN}(x))
\end{aligned}
$$

Read it out loud: "*x becomes x plus the attention of the normalized x.*" The residual connection (`x +`) is the main highway. Attention and FFN are on-ramps that **contribute** to the highway. This mental model — the **residual stream** — is the key to everything that follows.

### The FFN: SwiGLU, not ReLU

Inside each block, after attention, every token gets individually processed by a two-layer MLP. Classical transformers used ReLU. Modern LLMs use **SwiGLU** or **GELU**.

Plain FFN:
$$
\text{FFN}(x) = W_2 \cdot \text{ReLU}(W_1 x)
$$

SwiGLU (used by Llama, PaLM, Mixtral):
$$
\text{SwiGLU}(x) = (W_1 x \odot \sigma(W_g x)) \cdot W_2
$$

where $\sigma$ is the sigmoid and $\odot$ is elementwise multiplication. The gate $\sigma(W_g x)$ decides how much each hidden unit contributes. It's basically a learnable soft switch.

Why SwiGLU? Because empirically, when you burn $10^{24}$ FLOPs training a model, you want every free percentage point of loss improvement you can get. SwiGLU gives you about 0.5–1% perplexity over plain GELU. That's worth it when the training run costs millions.

In [ ]:
# --- LayerNorm ---
# PyTorch has nn.LayerNorm, but let's write it once so we know what's inside.

class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d))
        self.beta = nn.Parameter(torch.zeros(d))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        return self.gamma * (x - mean) / torch.sqrt(var + self.eps) + self.beta

# Sanity check: a normalized tensor should have mean ~0 and var ~1 per row.
ln = LayerNorm(16)
x = torch.randn(4, 16) * 5 + 3
y = ln(x)
print(f"input  mean={x.mean().item():+.3f}, std={x.std().item():.3f}")
print(f"output mean={y.mean().item():+.3f}, std={y.std().item():.3f}")

In [ ]:
# --- Multi-Head Self-Attention ---
# Causal mask so each position only sees the past. You've seen this in Part II;
# here it's written as one tight module.

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
        # Triangular mask. Register as buffer so it moves with the module.
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len)).view(1, 1, max_seq_len, max_seq_len)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)                                  # (B, T, 3C)
        q, k, v = qkv.split(C, dim=2)                      # each (B, T, C)
        # Reshape into heads: (B, nH, T, dH)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.drop(att)

        y = att @ v                                         # (B, nH, T, dH)
        y = y.transpose(1, 2).contiguous().view(B, T, C)    # (B, T, C)
        return self.proj(y)

In [ ]:
# --- SwiGLU FFN ---

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        # Two input projections for the gate + value, one output projection.
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up   = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


# --- The transformer block ---

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout=0.0, use_residual=True):
        super().__init__()
        self.ln1 = LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_seq_len, dropout=dropout)
        self.ln2 = LayerNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff)
        self.use_residual = use_residual   # knob for the break-things demo below

    def forward(self, x):
        if self.use_residual:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
        else:
            # Without residuals. Watch what happens.
            x = self.attn(self.ln1(x))
            x = self.ffn(self.ln2(x))
        return x

# Quick shape check
block = TransformerBlock(d_model=64, n_heads=4, d_ff=256, max_seq_len=32)
x = torch.randn(2, 16, 64)
print("input :", tuple(x.shape))
print("output:", tuple(block(x).shape))
print("params:", sum(p.numel() for p in block.parameters()))

## The residual stream view

Here is the mental model that will carry you through the rest of Part III and all of Part IV.

Imagine a vector $x_\ell \in \mathbb{R}^d$ at layer $\ell$. The block doesn't **replace** it — it **adds a correction**:

$$
x_{\ell+1} = x_\ell + \Delta_\ell
$$

where $\Delta_\ell$ is whatever attention and FFN cooked up. So the activation at layer 12 is literally:

$$
x_{12} = x_0 + \Delta_0 + \Delta_1 + \cdots + \Delta_{11}
$$

A sum. Not a composition. This is the **residual stream**, and interpretability research (Anthropic, etc.) treats it as a bus that every layer reads from and writes to. Different heads in different layers compute different features and dump them into the stream. The final layer norm + unembedding just projects this accumulated sum into vocab space.

Keep this picture in your head. It's why residuals are load-bearing.

## Break things on purpose: remove the residuals

Let's actually train two tiny stacks — one with residuals, one without — on a random regression task, and watch the loss.

In [ ]:
def train_toy(use_residual, steps=200, n_layers=6):
    torch.manual_seed(0)
    layers = nn.ModuleList([
        TransformerBlock(d_model=64, n_heads=4, d_ff=128,
                         max_seq_len=16, use_residual=use_residual)
        for _ in range(n_layers)
    ])
    model = nn.Sequential(*layers)
    head = nn.Linear(64, 1)
    opt = torch.optim.Adam(list(model.parameters()) + list(head.parameters()), lr=3e-4)

    # Toy task: predict the mean of the sequence from the last position.
    losses = []
    for step in range(steps):
        x = torch.randn(32, 16, 64)
        target = x.mean(dim=(1, 2), keepdim=False).unsqueeze(-1)
        h = model(x)
        pred = head(h[:, -1, :])
        loss = F.mse_loss(pred, target)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

print("Training WITH residuals ...")
loss_yes = train_toy(use_residual=True)
print("Training WITHOUT residuals ...")
loss_no = train_toy(use_residual=False)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(loss_yes, label="with residuals", color="#1f77b4", lw=2)
ax.plot(loss_no,  label="no residuals",   color="#d62728", lw=2)
ax.set_yscale("log")
ax.set_xlabel("step")
ax.set_ylabel("loss (log)")
ax.set_title("Residual connections: load-bearing, not decorative")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Notice how the no-residual version refuses to learn. The signal dies as it propagates through the stack, and the gradients die on the way back. This is the whole reason ResNet mattered in 2015 and the whole reason transformers work at depth.

You can make it more dramatic by increasing `n_layers=12` — a deep post-no-residual stack basically won't train at all.

---
# Section 2 — Positional Encoding

## The problem

Attention is a **set** operation. If you shuffle the tokens, attention doesn't care — it will produce the same set of outputs, just in a different order. "dog bites man" and "man bites dog" look identical to raw attention.

That's fine for a bag of words. It is catastrophic for language.

So we need to inject **position** somewhere. There are three main families.

### Family 1 — Sinusoidal (original, 2017)

Add a fixed (non-learned) vector to each token embedding, where the vector is built from sines and cosines of different frequencies:

$$
\text{PE}(pos, 2i) = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \quad
\text{PE}(pos, 2i+1) = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)
$$

Each dimension $i$ is a sinusoid with a different wavelength. Low $i$ → fast oscillation (fine-grained position). High $i$ → slow oscillation (coarse position). The model learns to read this fingerprint.

**Pro:** simple, no learned parameters.
**Con:** doesn't extrapolate well beyond trained length.

In [ ]:
def sinusoidal_pe(n_positions, d):
    pe = torch.zeros(n_positions, d)
    pos = torch.arange(n_positions).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

pe = sinusoidal_pe(128, 64)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(pe.numpy(), aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xlabel("embedding dimension")
ax.set_ylabel("position")
ax.set_title("Sinusoidal positional encoding\n(each row is the 'position fingerprint' for one token)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

You can see the structure immediately — low-index dimensions wiggle fast, high-index dimensions wiggle slow. The model learns to decode this.

### Family 2 — RoPE (Rotary Position Embedding)

RoPE, introduced in RoFormer and now used by Llama/Mistral/Qwen/basically everyone, is the elegant one.

Instead of *adding* to the embedding, RoPE **rotates** the query and key vectors by an angle proportional to their position. Split each head-dim into pairs, and rotate each pair by $\theta_i \cdot pos$:

$$
\text{RoPE}(q, pos) = R(\theta \cdot pos) \, q
$$

where $R$ is a block-diagonal 2D rotation. The key property:

$$
\langle R_{\theta m} q,\; R_{\theta n} k \rangle = \langle q,\; R_{\theta (n - m)} k \rangle
$$

The dot product depends only on the **relative** position $n - m$. Beautiful. It's also what lets RoPE models extrapolate to longer contexts (with some scaling tricks like NTK-aware interpolation, YaRN, etc.)

In [ ]:
def rope_angles(seq_len, d_head, base=10000.0):
    # frequencies for each pair of dims
    theta = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))  # (d_head/2,)
    pos = torch.arange(seq_len).float()                                     # (T,)
    freqs = torch.outer(pos, theta)                                         # (T, d_head/2)
    return freqs  # angles to rotate pair i at position t

def apply_rope(x, freqs):
    # x: (..., T, d_head), freqs: (T, d_head/2)
    cos = freqs.cos().unsqueeze(0)  # (1, T, d_head/2)
    sin = freqs.sin().unsqueeze(0)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    x_rot = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return x_rot.flatten(-2)

# Visualize: how does q.k vary with relative distance?
d_head = 64
T = 64
q = torch.randn(1, T, d_head)
k = q.clone()
freqs = rope_angles(T, d_head)
q_rot = apply_rope(q, freqs)
k_rot = apply_rope(k, freqs)

# Similarity of position t against every other position after RoPE
sim = (q_rot[0] @ k_rot[0].T) / math.sqrt(d_head)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(sim.numpy(), cmap="viridis")
ax.set_xlabel("key position")
ax.set_ylabel("query position")
ax.set_title("RoPE-rotated dot products\n(bright diagonal = strong near-neighbor preference)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

### Family 3 — ALiBi (Attention with Linear Biases)

ALiBi throws away positional embeddings entirely and adds a simple **linear penalty** directly to attention logits:

$$
\text{scores}_{ij} = q_i^\top k_j - m \cdot |i - j|
$$

Each head gets its own slope $m$ (a fixed geometric sequence like $1, 1/2, 1/4, 1/8, \ldots$). Further-away tokens get penalized more. That's it. No embeddings, no rotations.

ALiBi extrapolates well to longer context and is used in BLOOM, MPT, and a few others. RoPE won the mainstream though.

In [ ]:
def alibi_bias(n_heads, seq_len):
    # Geometric slopes: m_h = 2^(-8 h / n_heads)
    slopes = 2 ** (-8.0 * (torch.arange(1, n_heads + 1).float()) / n_heads)
    # distance matrix
    dist = torch.arange(seq_len).unsqueeze(0) - torch.arange(seq_len).unsqueeze(1)
    dist = dist.abs().float()
    # (n_heads, T, T)
    return -slopes.view(n_heads, 1, 1) * dist.unsqueeze(0)

bias = alibi_bias(n_heads=4, seq_len=32)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for h in range(4):
    axes[h].imshow(bias[h].numpy(), cmap="magma")
    axes[h].set_title(f"head {h}")
    axes[h].set_xlabel("key")
    if h == 0:
        axes[h].set_ylabel("query")
fig.suptitle("ALiBi biases — each head penalizes distance differently", y=1.02)
plt.tight_layout()
plt.show()

### TL;DR

| Scheme | Where it lives | Extrapolates? | Used by |
|---|---|---|---|
| Sinusoidal | Added to embeddings | Poorly | Original 2017 transformer |
| Learned abs. | Added to embeddings | No | GPT-2, BERT |
| **RoPE** | Rotates Q/K | With scaling tricks | Llama, Mistral, Qwen, GPT-NeoX |
| ALiBi | Biases logits | Naturally | BLOOM, MPT |

For our nanoGPT later, we'll use **learned absolute position embeddings** because they're the simplest thing that works, and training for 3 minutes on Shakespeare doesn't demand anything fancier.

---
# Section 3 — Encoder vs Decoder vs Encoder-Decoder

There are (historically) three architectural families. They differ almost entirely in **who is allowed to look at whom** — i.e. the attention mask.

### Encoder-only (BERT, RoBERTa, DeBERTa)

Bidirectional attention. Every token sees every other token. Trained with **masked language modeling**: randomly mask out 15% of tokens and predict them from the rest.

$$
\mathcal{L} = -\sum_{i \in \text{masked}} \log P(x_i \mid x_{\text{rest}})
$$

**Good at:** classification, embeddings, retrieval, NER.
**Bad at:** generation (no natural way to produce text left-to-right).

### Decoder-only (GPT, Llama, Mistral, Qwen, …)

Causal mask: token $i$ can only attend to tokens $\le i$. Trained with **next-token prediction**:

$$
\mathcal{L} = -\sum_t \log P(x_t \mid x_1, \ldots, x_{t-1})
$$

**Good at:** generation, few-shot prompting, scaling.
**Bad at:** classical bidirectional tasks out of the box — but in 2024+ it turned out they're fine at those too once you scale.

### Encoder-decoder (T5, BART, original Transformer)

Two stacks joined by **cross-attention**: the decoder attends to the encoder's outputs.

$$
\text{CrossAttn}(Q_{\text{dec}}, K_{\text{enc}}, V_{\text{enc}}) = \text{softmax}\!\left(\frac{Q_{\text{dec}} K_{\text{enc}}^\top}{\sqrt{d_k}}\right) V_{\text{enc}}
$$

The decoder's *queries* come from the decoder's own state, but the *keys and values* come from the encoder. It's how you translate: the decoder writes French while continuously glancing back at the English source.

**Good at:** translation, summarization, anything seq2seq with a clear input/output split.

In [ ]:
# Visualize the three mask patterns on a 12-token sequence.
T = 12

encoder_mask = torch.ones(T, T)
decoder_mask = torch.tril(torch.ones(T, T))
# Encoder-decoder: "prefix LM" — bidirectional on prefix, causal on the suffix
prefix_len = 5
prefix_lm_mask = torch.tril(torch.ones(T, T))
prefix_lm_mask[:prefix_len, :prefix_len] = 1.0

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, mask, title in zip(
    axes,
    [encoder_mask, decoder_mask, prefix_lm_mask],
    ["Encoder (BERT)\nbidirectional",
     "Decoder (GPT)\ncausal",
     "Prefix-LM (T5-ish)\nbi on prefix, causal after"]
):
    ax.imshow(mask.numpy(), cmap="Greens", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("key")
    ax.set_ylabel("query")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

### Why decoder-only won

Three reasons, roughly in order of importance:

1. **Simpler objective.** Next-token prediction is a single loss with a clean curriculum: every token in the training set contributes a gradient. MLM only learns from the 15% you masked. Per FLOP, next-token wins.

2. **Scaling laws.** Chinchilla-style scaling laws are cleanest for decoder-only. The field could tell, pretty early, that this is the architecture that keeps giving as you add compute.

3. **In-context learning.** Decoder-only models naturally handle *anything* as a prompt continuation task. Translation is "English: ... French:". Classification is "Review: ... Sentiment:". You don't need task-specific heads.

So while T5 and its kind are technically more general, they lost because the thing you want is the thing that scales best per FLOP. Everything from here on — nanoGPT, Llama, your training loop — will be decoder-only.

---
# Section 4 — Full Architecture Walkthrough

## Tracing a token from raw text to output

The big picture for any modern decoder-only LLM:

```
input_ids  (B, T)                      <- tokenizer output
    |
    v
token embeddings      (B, T, d)        <- nn.Embedding(vocab, d)
    +
position embeddings   (B, T, d)        <- nn.Embedding(max_T, d) or RoPE/sinusoidal
    |
    v
[Transformer Block] x N:
    x = x + MHA(LN(x))                 <- (B, T, d) throughout
    x = x + FFN(LN(x))
    |
    v
final LayerNorm       (B, T, d)
    |
    v
lm_head (Linear)      (B, T, vocab)    <- project to vocab
    |
    v
softmax -> next-token distribution     <- (B, T, vocab)
```

Crucially, **the shape `(B, T, d)` is preserved through every block**. That's what makes the residual stream a clean abstraction: it's just the same vector getting updated.

Let's pick some concrete numbers and compute the parameter budget of a 7B model.

In [ ]:
# Param counts for a generic Llama-2 7B style model
@dataclass
class LlamaConfig:
    vocab: int = 32000
    d_model: int = 4096
    n_layers: int = 32
    n_heads: int = 32
    d_ff: int = 11008        # SwiGLU expansion ~= 2.7 * d_model
    max_seq: int = 4096

cfg = LlamaConfig()

# 1. Embeddings
embed_params = cfg.vocab * cfg.d_model

# 2. Per block: attention (q, k, v, o) + FFN SwiGLU (gate, up, down) + 2 LN
attn_per_block = 4 * cfg.d_model * cfg.d_model
ffn_per_block  = 3 * cfg.d_model * cfg.d_ff
ln_per_block   = 2 * cfg.d_model  # 2 layernorms, gamma only (RMSNorm)
block_params = attn_per_block + ffn_per_block + ln_per_block

# 3. Final LN + lm_head (often tied with embeddings; here assume untied for clarity)
head_params = cfg.d_model + cfg.vocab * cfg.d_model  # LN gamma + linear

total_blocks = cfg.n_layers * block_params
total = embed_params + total_blocks + head_params

def fmt(n):
    for unit in ["", "K", "M", "B"]:
        if abs(n) < 1000:
            return f"{n:6.2f}{unit}"
        n /= 1000
    return f"{n:.2f}T"

print("Llama-2 7B (approx) parameter breakdown")
print("-" * 46)
print(f"Embeddings            : {fmt(embed_params):>10}  ({100*embed_params/total:5.1f}%)")
print(f"Attention (all blocks): {fmt(cfg.n_layers * attn_per_block):>10}  ({100*cfg.n_layers*attn_per_block/total:5.1f}%)")
print(f"FFN       (all blocks): {fmt(cfg.n_layers * ffn_per_block):>10}  ({100*cfg.n_layers*ffn_per_block/total:5.1f}%)")
print(f"LayerNorms            : {fmt(cfg.n_layers * ln_per_block):>10}  ({100*cfg.n_layers*ln_per_block/total:5.1f}%)")
print(f"LM head + final LN    : {fmt(head_params):>10}  ({100*head_params/total:5.1f}%)")
print("-" * 46)
print(f"TOTAL                 : {fmt(total):>10}")

In [ ]:
# Pie chart of where the parameters live
fig, ax = plt.subplots(figsize=(6.5, 6.5))
labels = ["Embeddings", "Attention", "FFN (SwiGLU)", "LM head"]
sizes = [embed_params,
         cfg.n_layers * attn_per_block,
         cfg.n_layers * ffn_per_block,
         head_params]
colors = ["#8da0cb", "#fc8d62", "#66c2a5", "#e78ac3"]
ax.pie(sizes, labels=labels, colors=colors, autopct="%1.1f%%",
       startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 2})
ax.set_title("Where do 7B parameters live?\n(FFN dominates, not attention)")
plt.tight_layout()
plt.show()

### The surprise

People assume attention is where the action is. But **~65% of the parameters are in the FFN** and only ~30% in attention. And it gets worse with SwiGLU, which has three matrices instead of two.

This matters for interpretability: mechanistic interpretability work has found that a lot of **factual knowledge** lives in FFN layers — the attention modules are closer to a routing / information-mixing mechanism, and the FFN acts like a giant key-value memory. Attention is the postal service; FFN is the warehouse.

## Building nanoGPT — the real one

Let's now write a complete, self-contained tiny GPT. This is the model we'll actually train in Section 5 and generate with in Section 6.

Specs:
- `vocab_size` = whatever our tokenizer gives us (character-level)
- `d_model = 128`
- `n_heads = 4`
- `n_layers = 4`
- `block_size = 128`
- Learned absolute position embeddings (simplest thing)
- GELU FFN (simpler than SwiGLU for a toy)
- Tied embeddings (`lm_head.weight = token_embed.weight`)

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int = 128
    n_layers: int = 4
    n_heads: int = 4
    d_model: int = 128
    d_ff: int = 512
    dropout: float = 0.0


class GELU_FFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.block_size, cfg.dropout)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = GELU_FFN(cfg.d_model, cfg.d_ff, cfg.dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class NanoGPT(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        # Weight tying: save vocab*d_model params + helps training
        self.lm_head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, f"sequence length {T} > block_size {self.cfg.block_size}"
        pos = torch.arange(T, device=idx.device).unsqueeze(0)   # (1, T)
        x = self.tok_emb(idx) + self.pos_emb(pos)               # (B, T, d)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                # (B, T, vocab)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def num_params(self):
        # tied params — count only once
        n = sum(p.numel() for p in self.parameters())
        return n

In [ ]:
# Sanity check the model with a fake vocab
fake_cfg = GPTConfig(vocab_size=100)
m = NanoGPT(fake_cfg)
print(f"nanoGPT params: {m.num_params()/1e3:.1f}K")

idx = torch.randint(0, 100, (2, 32))
logits, _ = m(idx)
print("input  :", tuple(idx.shape))
print("logits :", tuple(logits.shape))
targets = torch.randint(0, 100, (2, 32))
_, loss = m(idx, targets)
print(f"untrained loss = {loss.item():.3f}  (random would be {math.log(100):.3f})")

A freshly-initialized model with `vocab=100` should have loss near $\ln(100) \approx 4.6$ because its predictions are uniform over the vocab. Ours is close but not exact because of the init scale. Good. Let's go train it.

---
# Section 5 — Training: Making It Learn

Now we make it real. We'll download Tiny Shakespeare, build a character-level tokenizer (no BPE today — we want to train fast), and run a training loop that should reduce loss from ~4.2 to ~1.5 in a couple of minutes on CPU.

## The objective

For every position $t$ in a sequence, the model outputs a distribution $P(\cdot \mid x_{<t})$. The target is $x_t$ itself. Cross-entropy loss:

$$
\mathcal{L} = -\frac{1}{T} \sum_{t=1}^{T} \log P_\theta(x_t \mid x_{1:t-1})
$$

Plain English: *how surprised was the model by each actual next-token*. Lower = better. The exponent of this (base $e$) is called **perplexity** and has the interpretation "the effective number of equally-likely options the model thinks it's choosing from."

### Training step

1. Sample a batch of chunks `(B, T+1)` from the corpus.
2. Split: `x = chunk[:, :-1]`, `y = chunk[:, 1:]` (shifted by one).
3. Forward pass → logits `(B, T, V)`.
4. Cross-entropy against `y`.
5. `loss.backward()` — gradients flow backward through the entire transformer stack. The residual connections make this tractable.
6. Optimizer step.

### Why Adam, not SGD?

Plain SGD uses a single learning rate for all parameters. But transformers have parameters of wildly different scales — the attention QKV matrices evolve very differently from the LayerNorm gains. Adam (and its cousin AdamW) maintains **per-parameter** running estimates of the first and second moments of the gradient:

$$
m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t, \quad
v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2
$$

$$
\theta_{t+1} = \theta_t - \eta \cdot \frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon}
$$

Each parameter gets its own effective step size. The difference between Adam and AdamW is whether weight decay is applied in a clean, decoupled way (W) or fused into the gradient (original Adam). Always use AdamW for transformers.

### Learning rate schedule: warmup + cosine decay

Transformers at initialization are **fragile**. If you slam them with a big learning rate on step 1, loss goes to NaN. So we warm up: linearly increase LR from 0 to peak over the first ~500–2000 steps. Then cosine-decay down to something like 10% of peak over the rest of training.

In [ ]:
# Download Tiny Shakespeare (~1 MB)
import os, urllib.request

DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = "tiny_shakespeare.txt"

if not os.path.exists(DATA_PATH):
    print("Downloading tiny shakespeare ...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()

print(f"Corpus: {len(text):,} characters")
print("First 200 chars:")
print(text[:200])

In [ ]:
# Character-level tokenizer. Not sexy, but it trains in seconds.
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}

def encode(s): return [stoi[c] for c in s]
def decode(ids): return "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

print(f"vocab size: {vocab_size}")
print(f"train: {len(train_data):,} tokens, val: {len(val_data):,} tokens")

In [ ]:
# Batch sampler: grabs random chunks of length block_size + 1
def get_batch(split, block_size, batch_size):
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[i:i+block_size]       for i in ix])
    y = torch.stack([source[i+1:i+1+block_size]   for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train", 64, 4)
print("x shape:", tuple(xb.shape))
print("y shape:", tuple(yb.shape))
print("\nfirst sample, first 60 chars:")
print("x:", repr(decode(xb[0, :60].tolist())))
print("y:", repr(decode(yb[0, :60].tolist())))
print("note: y is x shifted by one — that's the whole next-token trick")

In [ ]:
# Warmup + cosine schedule
def lr_lambda(step, warmup, total, min_ratio=0.1):
    if step < warmup:
        return step / max(1, warmup)
    # cosine decay from 1 to min_ratio
    progress = (step - warmup) / max(1, total - warmup)
    return min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * progress))

# Visualize the schedule for our run
steps_total = 2000
warmup = 100
ys = [lr_lambda(s, warmup, steps_total) for s in range(steps_total)]
plt.figure(figsize=(9, 3))
plt.plot(ys, color="#2ca02c", lw=2)
plt.axvline(warmup, color="k", ls="--", alpha=0.4, label=f"warmup end ({warmup})")
plt.title("Learning rate schedule (warmup + cosine decay)")
plt.xlabel("step")
plt.ylabel("LR multiplier")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Build the model we'll actually train
cfg = GPTConfig(
    vocab_size=vocab_size,
    block_size=128,
    n_layers=4,
    n_heads=4,
    d_model=128,
    d_ff=512,
    dropout=0.1,
)
model = NanoGPT(cfg).to(device)
print(f"nanoGPT has {model.num_params()/1e6:.2f}M parameters")
print(f"Device: {device}")

In [ ]:
# Training loop
@torch.no_grad()
def estimate_loss(batches=20):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(batches):
            xb, yb = get_batch(split, cfg.block_size, 32)
            _, loss = model(xb, yb)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out


@torch.no_grad()
def quick_sample(prompt="ROMEO:", max_new=120, temperature=0.8):
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new):
        idx_cond = idx[:, -cfg.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return decode(idx[0].tolist())


# hyperparams
max_steps   = 2000
eval_every  = 250
sample_every = 500
batch_size  = 64
peak_lr     = 3e-3
warmup_steps = 100

optimizer = torch.optim.AdamW(model.parameters(), lr=peak_lr, betas=(0.9, 0.95), weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=lambda s: lr_lambda(s, warmup_steps, max_steps)
)

history = {"step": [], "train": [], "val": [], "lr": []}
samples_at_steps = {}

t0 = time.time()
for step in range(max_steps + 1):
    if step % eval_every == 0:
        losses = estimate_loss()
        history["step"].append(step)
        history["train"].append(losses["train"])
        history["val"].append(losses["val"])
        history["lr"].append(optimizer.param_groups[0]["lr"])
        print(f"step {step:4d} | train {losses['train']:.4f} | val {losses['val']:.4f} "
              f"| lr {optimizer.param_groups[0]['lr']:.2e} | t {time.time()-t0:5.1f}s")

    if step % sample_every == 0:
        samples_at_steps[step] = quick_sample(prompt="ROMEO:", max_new=120, temperature=0.8)

    if step == max_steps:
        break

    xb, yb = get_batch("train", cfg.block_size, batch_size)
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

print(f"\nTotal training time: {time.time()-t0:.1f}s")

In [ ]:
# Loss curves
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history["step"], history["train"], label="train", lw=2, color="#1f77b4", marker="o")
ax.plot(history["step"], history["val"],   label="val",   lw=2, color="#ff7f0e", marker="s")
ax.set_xlabel("step")
ax.set_ylabel("cross-entropy loss")
ax.set_title("nanoGPT training on Shakespeare")
ax.grid(alpha=0.3)
ax.legend()

# Twin axis for LR
ax2 = ax.twinx()
ax2.plot(history["step"], history["lr"], color="#2ca02c", ls="--", alpha=0.5, label="lr")
ax2.set_ylabel("learning rate", color="#2ca02c")
plt.tight_layout()
plt.show()

print(f"Starting val loss: {history['val'][0]:.3f}  (random baseline = ln({vocab_size}) = {math.log(vocab_size):.3f})")
print(f"Final val loss   : {history['val'][-1]:.3f}")
print(f"Final perplexity : {math.exp(history['val'][-1]):.2f}")

## Samples through training

Watch what happens to the generations as the model learns. Early checkpoints are garbage. By the end it's producing something that looks eerily Shakespearean.

In [ ]:
for step, sample in samples_at_steps.items():
    print(f"--- step {step} ---")
    print(sample)
    print()

The model went from random character soup to something with line breaks, character names, iambic-ish cadence, and words that at least *look* English. From a 300K-parameter model trained for ~1 minute. This is why people are excited about transformers.

---
# Section 6 — Inference: Generating Text

Training is over. Now we generate.

## The autoregressive loop

```
prompt = "ROMEO:"
for _ in range(max_new_tokens):
    logits = model(prompt)        # (B, T, vocab)
    last   = logits[:, -1, :]     # we only care about the LAST position
    probs  = softmax(last / T)    # turn into a distribution
    next_id = sample(probs)       # pick one
    prompt = prompt + next_id     # append
```

That's the entire thing. Generation is a tight loop of forward passes where we only ever care about the last token's logits.

**Important subtlety:** the model is stateless. Every iteration we re-run the entire sequence through all layers. That's $O(T^2)$ attention total. Part IV will show you how to fix this with the **KV cache** — but for now, accept the wastefulness.

## Four sampling strategies

Given logits $\ell \in \mathbb{R}^V$ at the last position, how do we choose a token?

### 1. Greedy (argmax)

$$
x_{t+1} = \arg\max_v \ell_v
$$

Deterministic. Boring. Loops. "the the the the the" is a real failure mode.

### 2. Temperature sampling

$$
P(x_{t+1} = v) = \text{softmax}(\ell / T)_v
$$

- $T \to 0$: converges to greedy.
- $T = 1$: the model's "native" distribution.
- $T > 1$: flattens the distribution — more diverse, less coherent.
- $T = \infty$: uniform. Gibberish.

### 3. Top-k

Keep only the $k$ most-probable tokens, zero out the rest, renormalize, sample. Prevents the long tail of garbage from ever being chosen, but $k$ is static — not responsive to how sharp the distribution actually is.

### 4. Top-p (nucleus) sampling

Sort tokens by probability descending. Keep the smallest set whose cumulative probability $\ge p$. Renormalize. Sample.

This adapts: when the model is confident (one token at 95%), you draw from a tiny set. When the model is uncertain, you draw from a wider set. **Top-p is what you should default to** for most generation today, usually $p = 0.9$ with $T = 0.8$.

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=200, temperature=1.0, top_k=None, top_p=None):
    # Full sampling function supporting greedy, temperature, top-k, top-p.
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -cfg.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / max(temperature, 1e-8)

        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float("-inf")

        if top_p is not None:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            sorted_probs = F.softmax(sorted_logits, dim=-1)
            cum = torch.cumsum(sorted_probs, dim=-1)
            # mask tokens beyond the nucleus (but always keep at least one)
            remove = cum > top_p
            remove[..., 1:] = remove[..., :-1].clone()
            remove[..., 0] = False
            sorted_logits[remove] = float("-inf")
            # scatter back to original ordering
            logits = torch.full_like(logits, float("-inf"))
            logits.scatter_(1, sorted_idx, sorted_logits)

        if temperature < 1e-6:  # numerically "greedy"
            next_id = logits.argmax(dim=-1, keepdim=True)
        else:
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        idx = torch.cat([idx, next_id], dim=1)
    return decode(idx[0].tolist())

In [ ]:
prompt = "ROMEO:"
print("=" * 60)
print("GREEDY (T=0)")
print("=" * 60)
print(generate(model, prompt, max_new_tokens=200, temperature=0.0))
print()

In [ ]:
print("=" * 60)
print("TEMPERATURE 0.5 (safe, coherent)")
print("=" * 60)
print(generate(model, prompt, max_new_tokens=200, temperature=0.5))
print()

print("=" * 60)
print("TEMPERATURE 1.0 (model's native distribution)")
print("=" * 60)
print(generate(model, prompt, max_new_tokens=200, temperature=1.0))
print()

print("=" * 60)
print("TEMPERATURE 1.8 (unhinged)")
print("=" * 60)
print(generate(model, prompt, max_new_tokens=200, temperature=1.8))

In [ ]:
print("=" * 60)
print("TOP-K = 10 (keep only the 10 most likely next tokens)")
print("=" * 60)
print(generate(model, prompt, max_new_tokens=200, temperature=1.0, top_k=10))
print()

print("=" * 60)
print("TOP-P = 0.9 (nucleus sampling — the practical default)")
print("=" * 60)
print(generate(model, prompt, max_new_tokens=200, temperature=0.8, top_p=0.9))

## Visualizing the distribution under each strategy

Let's take a single prediction from the trained model and see how the different strategies reshape it.

In [ ]:
@torch.no_grad()
def get_next_token_probs(prompt):
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    logits, _ = model(idx)
    return logits[0, -1, :].cpu()

logits = get_next_token_probs("ROMEO:\nBut soft, what light through yonder window break")

def reshape_topk(logits, k):
    probs = F.softmax(logits.clone(), dim=-1)
    topk_vals, topk_idx = torch.topk(probs, k)
    out = torch.zeros_like(probs)
    out[topk_idx] = topk_vals / topk_vals.sum()
    return out

def reshape_topp(logits, p):
    probs = F.softmax(logits.clone(), dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cum = torch.cumsum(sorted_probs, dim=-1)
    mask = cum <= p
    mask[0] = True
    kept_probs = sorted_probs * mask
    kept_probs = kept_probs / kept_probs.sum()
    out = torch.zeros_like(probs)
    out[sorted_idx] = kept_probs
    return out

def reshape_temp(logits, T):
    return F.softmax(logits / T, dim=-1)

# Top 20 tokens by raw probability for the plot axis
base_probs = F.softmax(logits, dim=-1)
top20 = torch.topk(base_probs, 20)
idx20 = top20.indices
labels = [repr(itos[i.item()]) for i in idx20]

variants = {
    "T=1.0 (raw)"     : reshape_temp(logits, 1.0)[idx20],
    "T=0.5 (sharp)"   : reshape_temp(logits, 0.5)[idx20],
    "T=2.0 (flat)"    : reshape_temp(logits, 2.0)[idx20],
    "top-k=5"         : reshape_topk(logits, 5)[idx20],
    "top-p=0.9"       : reshape_topp(logits, 0.9)[idx20],
}

fig, axes = plt.subplots(len(variants), 1, figsize=(12, 9), sharex=True)
for ax, (name, probs) in zip(axes, variants.items()):
    ax.bar(range(len(labels)), probs.numpy(), color="#4c72b0")
    ax.set_ylabel(name, rotation=0, ha="right", va="center")
    ax.set_ylim(0, max(0.05, probs.max().item() * 1.1))
    ax.grid(alpha=0.3, axis="y")
axes[-1].set_xticks(range(len(labels)))
axes[-1].set_xticklabels(labels, rotation=45)
axes[0].set_title("How different sampling strategies reshape the next-token distribution")
plt.tight_layout()
plt.show()

Look at the bottom two: top-k clips to exactly 5 tokens regardless of their probability mass. Top-p adaptively picks however many tokens are needed to cover 90% of the mass. When the model is confident, that might be 1 token; when uncertain, 30.

## The KV cache: a preview

Quick pitch for Part IV. Notice that in our generation loop, every new token forces us to re-run *all previous tokens* through the full stack. But the Keys and Values for the previous tokens **don't change** — we already computed them. We should just cache them.

With a KV cache:
- Step 1 is still $O(T^2)$ (we process the prompt).
- Each subsequent step is $O(T)$ instead of $O(T^2)$.
- Memory grows linearly with context length to store the cache.

For a 70B model at 4K context in fp16, the KV cache is roughly **10 GB** — often more than the activations. This is why long context is expensive, and why 2024–2026 has been a parade of tricks (MQA, GQA, quantized KV cache, paged attention, sliding window attention, linear attention, etc.) to shrink it. All of Part IV.

---
# Wrap-up

You built it. All of it.

- ✅ A transformer block (LayerNorm + MHA + FFN + residuals)
- ✅ Three positional encoding schemes with heatmaps
- ✅ A full architecture walkthrough with param accounting
- ✅ A trained, working nanoGPT on Shakespeare
- ✅ Four sampling strategies with live demos

The code in this notebook is about 300 lines of PyTorch. The same 300 lines, scaled up with more layers, more data, more compute, and a better tokenizer, is what every frontier lab is running. There is nothing else secret in the architecture. The difficulty lives in the training infrastructure, the data, the scale, and the alignment — which is what Parts IV through VIII are about.

### Checkpoint quiz

1. Why does pre-norm train more stably than post-norm at depth?
2. What goes wrong if you remove residual connections from a 12-layer transformer?
3. In a 7B Llama-style model, which component has the most parameters — attention, FFN, or embeddings? Roughly what fraction?
4. What does RoPE do that sinusoidal encoding doesn't, mathematically?
5. Why do we need a learning-rate warmup for transformers?
6. What's the relationship between cross-entropy loss and perplexity?
7. What fails about greedy decoding?
8. Top-k vs top-p: why is top-p adaptive and top-k not?
9. Rough operation count for decoding 100 tokens from a prompt of 1000, without a KV cache vs with?

### What's next

**Part IV — Scaling & Efficiency:** the KV cache, Flash Attention, quantization (INT8, FP8, TurboQuant, RotorQuant), MoE, speculative decoding. This is where you learn how modern serving actually works at 100+ tokens/sec on a single GPU.